In [30]:
using Pkg
Pkg.activate("C:/Users/selha/Desktop/MAAAI/environment")  # <-- CAMBIAR A TU RUTA DE ENV

  Activating project at `C:\Users\selha\Desktop\MAAAI\environment`


In [31]:
Pkg.instantiate()   # solo necesario la primera vez 

In [61]:
using CSV, DataFrames, Statistics, Random
using MLJ
using MLJModels
using MLJModelInterface
using MLJBase
import MLJBase: transform
using DataFramesMeta
using Plots
using Glob
using MLJScikitLearnInterface

## 1. Carga y unificación de datos

Los datos originales están distribuidos en múltiples ficheros CSV, uno por cada sujeto.  
El objetivo de este bloque es:

- Buscar todos los CSV dentro del directorio raíz.
- Leerlos en memoria de forma homogénea.
- Concatenarlos en un único DataFrame consolidado.
- Guardar `dataset_consolidado.csv` para usarlo en el resto de la práctica.


In [19]:
# Ruta donde están los CSV originales
DATA_ROOT = "C:\\Users\\selha\\Desktop\\MAAAI\\datasets"   # <-- CAMBIAR A TU RUTA DE DATASETS

# Crear carpeta donde guardaremos todos los outputs del preprocesado
mkpath("data_processed")

# Buscar todos los CSV dentro de la carpeta (incluye subcarpetas)
csv_files = Glob.glob("**/*.csv", DATA_ROOT)

println("Archivos encontrados: ", length(csv_files))

# Leer todos los CSV y almacenarlos en un array de DataFrames
dfs = DataFrame[]

for file in csv_files
    try
        df_tmp = CSV.read(file, DataFrame; normalizenames=true)
        push!(dfs, df_tmp)
        println("Leído: ", basename(file), "  (", nrow(df_tmp), " filas)")
    catch e
        @warn "No se pudo leer el CSV: $file" exception=(e, catch_backtrace())
    end
end

# Concatenación final
df = vcat(dfs...)
println("Dataset final: ", nrow(df), " filas y ", ncol(df), " columnas.")

# Guardar dataset consolidado
CSV.write("data_processed/dataset_consolidado.csv", df)
println("Guardado dataset consolidado.")


Archivos encontrados: 15
Leído: Sujeto_02.csv  (302 filas)
Leído: Sujeto_04.csv  (317 filas)
Leído: Sujeto_06.csv  (325 filas)
Leído: Sujeto_08.csv  (281 filas)
Leído: Sujeto_10.csv  (294 filas)
Leído: Sujeto_12.csv  (320 filas)
Leído: Sujeto_14.csv  (323 filas)
Leído: Sujeto_16.csv  (366 filas)
Leído: Sujeto_18.csv  (364 filas)
Leído: Sujeto_20.csv  (354 filas)
Leído: Sujeto_22.csv  (321 filas)
Leído: Sujeto_24.csv  (381 filas)
Leído: Sujeto_26.csv  (392 filas)
Leído: Sujeto_28.csv  (382 filas)
Leído: Sujeto_30.csv  (383 filas)
Dataset final: 5105 filas y 563 columnas.
Guardado dataset consolidado.


## 2. Resumen del conjunto de datos

En esta sección obtenemos una descripción básica del dataset consolidado:

- Número total de instancias (filas)
- Número total de variables
- Número de individuos (`subject`)
- Número de clases de salida (`Activity`)


In [21]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

num_variables_totales = ncol(df)
num_instancias = nrow(df)

# Número de individuos
if "subject" in names(df)
    num_individuos = length(unique(df.subject))
else
    @warn "No se encontró la columna 'subject'; no se puede calcular el número de individuos."
    num_individuos = missing
end

# Número de clases de salida
if "Activity" in names(df)
    num_clases_salida = length(unique(df.Activity))
else
    @warn "No se encontró la columna 'Activity'; no se puede calcular el número de clases de salida."
    num_clases_salida = missing
end

# Variables de entrada (todas excepto subject + Activity)
num_features = num_variables_totales - 2

println("\n=== Resumen del dataset ===")
println("Número total de variables (incluyendo subject y Activity): ", num_variables_totales)
println("Número de variables de características: ", num_features)
println("Número de instancias: ", num_instancias)
println("Número de individuos: ", num_individuos)
println("Número de clases de salida: ", num_clases_salida)
println("=============================================")


=== Resumen del dataset ===
Número total de variables (incluyendo subject y Activity): 563
Número de variables de características: 561
Número de instancias: 5105
Número de individuos: 15
Número de clases de salida: 6


## 3. Análisis de valores ausentes

En esta sección calculamos:

- El **porcentaje de valores nulos por variable**  
- El **porcentaje total de valores nulos** en el dataset  

Esto permite entender la magnitud del problema de valores faltantes y justificar
posteriormente el método de imputación empleado (subject-wise con media/mediana).

In [22]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

# Análisis de valores ausentes
n_rows = nrow(df)
n_cols = ncol(df)

# Porcentaje de nulos por columna
porc_nulos_col = Dict{String, Float64}()

for col in names(df)
    n_missing = count(ismissing, df[!, col])
    porc_nulos_col[col] = 100 * n_missing / n_rows
end

# Porcentaje total de nulos en todo el dataset
total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
total_values = n_rows * n_cols
porc_total_missing = 100 * total_missing / total_values

println("=== Porcentaje de valores nulos por columna ===")
for (col, pct) in sort(collect(porc_nulos_col); by = x -> x[2], rev = true)
    println(rpad(col, 30), ": ", round(pct, digits = 2), "%")
end

println("\nPorcentaje total de valores nulos en el dataset: ",
        round(porc_total_missing, digits = 2), "%")
println("===============================================")

=== Porcentaje de valores nulos por columna ===
fBodyGyro_bandsEnergy_49_56_2 : 10.85%
tBodyAcc_arCoeff_Z_2          : 10.79%
tBodyGyro_arCoeff_Y_2         : 10.54%
tGravityAcc_mad_Z             : 10.5%
fBodyAccJerk_bandsEnergy_25_32: 10.46%
tBodyGyroJerkMag_arCoeff_1    : 10.46%
fBodyGyro_bandsEnergy_25_32_2 : 10.42%
tBodyAcc_correlation_X_Y      : 10.38%
fBodyAcc_maxInds_Y            : 10.32%
tGravityAccMag_std_           : 10.32%
tGravityAccMag_entropy_       : 10.32%
fBodyGyro_maxInds_Z           : 10.3%
tBodyAccJerk_energy_X         : 10.28%
fBodyAcc_mad_Y                : 10.26%
fBodyAcc_bandsEnergy_25_32_2  : 10.26%
fBodyBodyGyroMag_iqr_         : 10.24%
fBodyGyro_bandsEnergy_49_56_1 : 10.24%
tBodyGyroMag_mad_             : 10.24%
fBodyAccJerk_meanFreq_Y       : 10.24%
tGravityAcc_std_Y             : 10.21%
fBodyAccJerk_mean_X           : 10.19%
tBodyGyro_mean_Z              : 10.19%
tBodyGyroJerk_arCoeff_Z_3     : 10.17%
tBodyGyroJerk_arCoeff_Z_2     : 10.15%
fBodyAcc_kurtosis_

## 4. Imputación de valores ausentes (subject-wise)

En esta sección imputamos los valores faltantes de las variables numéricas siguiendo
un criterio **por sujeto**:

- Para cada sujeto (`subject`) se toman únicamente sus propias observaciones.
- Para cada columna numérica con valores ausentes:
  - Se comprueba si hay outliers mediante el rango intercuartílico (IQR).
  - Si **hay outliers**, se imputa con la **mediana**.
  - Si **no hay outliers**, se imputa con la **media**.
- No se modifican las columnas `subject` ni `Activity`.

El resultado es un nuevo dataset imputado que conserva la estructura original pero sin
valores faltantes en las variables numéricas.


In [23]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)


# -----------------------------------------------------------
# Función auxiliar para detectar columnas numéricas (permitiendo Missing)
# -----------------------------------------------------------
function col_contains_numeric(eltyp)
    if eltyp <: Real
        return true
    end
    try
        for t in Base.uniontypes(eltyp)
            if t <: Real
                return true
            end
        end
    catch
    end
    return false
end

# -----------------------------------------------------------
# Detección de outliers con IQR
# -----------------------------------------------------------
function tiene_outliers(vals)
    q1 = quantile(vals, 0.25)
    q3 = quantile(vals, 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    any(x -> x < lower || x > upper, vals)
end

# -----------------------------------------------------------
# Imputación subject-wise
# -----------------------------------------------------------
function impute_subjectwise(df::DataFrame)
    df_imp = deepcopy(df)

    excluded = Set(["subject", "Activity"])   # no se imputan estas columnas

    # Detectar columnas numéricas a imputar
    numeric_cols = String[]
    for c in names(df_imp)
        if c ∉ excluded && col_contains_numeric(eltype(df_imp[!, c]))
            push!(numeric_cols, c)
        end
    end

    subjects = unique(df_imp.subject)

    println("Columnas numéricas a imputar: ", length(numeric_cols))
    println("Sujetos encontrados: ", length(subjects))

    for s in subjects
        rows_subject = df_imp.subject .== s

        for col in numeric_cols
            colvec = df_imp[rows_subject, col]
            nmiss = count(ismissing, colvec)
            if nmiss == 0
                continue
            end

            nonmiss = collect(skipmissing(colvec))
            if isempty(nonmiss)
                continue
            end

            method = tiene_outliers(nonmiss) ? "median" : "mean"
            value  = method == "median" ? median(nonmiss) : mean(nonmiss)

            mask = rows_subject .& ismissing.(df_imp[!, col])
            df_imp[mask, col] .= value
        end
    end

    return df_imp
end


# -----------------------------------------------------------
# Aplicar imputación y guardar resultado
# -----------------------------------------------------------
df_imputed = impute_subjectwise(df)

println("\nDataset imputado: ", nrow(df_imputed), " filas, ", ncol(df_imputed), " columnas.")

CSV.write("data_processed/dataset_consolidado_imputed.csv", df_imputed)

println("Guardado en: data_processed/dataset_consolidado_imputed.csv")


Columnas numéricas a imputar: 561
Sujetos encontrados: 15

Dataset imputado: 5105 filas, 563 columnas.
Guardado en: data_processed/dataset_consolidado_imputed.csv


## 5. Partición holdout (10 % de sujetos)

En este bloque se reserva un **10 % de los sujetos completos** como conjunto de
**test final (holdout)**, siguiendo las indicaciones del enunciado:

- Se parte del dataset ya imputado (`df_imputed`).
- Se obtienen todos los identificadores de sujetos (`subject`).
- Con la semilla `104` se selecciona aleatoriamente el 10 % de los sujetos.
- Todas las filas de esos sujetos pasan a formar el conjunto **test**.
- El resto de sujetos componen el conjunto **train**.

Este conjunto de test **no se utiliza en la validación cruzada** y se reserva
exclusivamente para la evaluación final de los modelos seleccionados.

In [24]:
# Cargar dataset imputado desde data_processed
df_imputed = CSV.read("data_processed/dataset_consolidado_imputed.csv", DataFrame)

println("Sujetos detectados en el dataset imputado:")
subjects = unique(df_imputed.subject)
println(subjects)

# Semilla pedida en el enunciado
Random.seed!(104)

# 10% de sujetos → al menos 1
n_test = max(1, round(Int, length(subjects) * 0.10))

println("Número total de sujetos: ", length(subjects))
println("Número de sujetos para TEST (10%): ", n_test)

# Selección aleatoria reproducible
test_subjects = Random.shuffle(subjects)[1:n_test]

println("\n=== Sujetos seleccionados para TEST (holdout) ===")
println(test_subjects)

# Máscaras de pertenencia
test_mask  = in.(df_imputed.subject, Ref(test_subjects))
train_mask = .!test_mask

# Particionar
df_test  = df_imputed[test_mask, :]
df_train = df_imputed[train_mask, :]

println("\nFilas train: ", nrow(df_train))
println("Filas test : ", nrow(df_test))

# ------------------ Guardado en data_processed ------------------

CSV.write("data_processed/dataset_train.csv", df_train)
CSV.write("data_processed/dataset_test.csv", df_test)

# Lista de sujetos del test
ts_df = DataFrame(subject = test_subjects)
CSV.write("data_processed/test_subjects.csv", ts_df)

# Informe del número de muestras por sujeto en el dataset completo
counts = combine(groupby(df_imputed, :subject), nrow => :n_rows)
CSV.write("data_processed/subject_counts.csv", counts)

println("\nArchivos guardados en data_processed/:")
println(" - Train dataset:        dataset_train.csv")
println(" - Test dataset:         dataset_test.csv")
println(" - Test subjects list:   test_subjects.csv")
println(" - Subject counts:       subject_counts.csv")


Sujetos detectados en el dataset imputado:
[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Número total de sujetos: 15
Número de sujetos para TEST (10%): 2

=== Sujetos seleccionados para TEST (holdout) ===
[16, 8]

Filas train: 4458
Filas test : 647

Archivos guardados en data_processed/:
 - Train dataset:        dataset_train.csv
 - Test dataset:         dataset_test.csv
 - Test subjects list:   test_subjects.csv
 - Subject counts:       subject_counts.csv


## 6. Validación cruzada individual-wise (5-Fold)

Tras aplicar la partición *holdout*, usamos únicamente el conjunto de entrenamiento
(`df_train`) para construir una validación cruzada 5-fold basada en **sujetos**:

- Cada fold contiene un subconjunto de sujetos completos.
- En cada fold, uno (o varios) sujetos se usan como **test interno**.
- El resto se usan como **train**.
- Nunca se mezclan instancias de un mismo sujeto entre train y test.
- Se fija la semilla `104` para reproducibilidad.

Este esquema es imprescindible porque los datos están fuertemente correlacionados por
sujeto; por tanto, una validación aleatoria estándar produciría *data leakage*.

In [25]:
# Cargar el conjunto de entrenamiento generado en el holdout
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)

# Sujetos disponibles en TRAIN
subjects_train = unique(df_train.subject)
n_subjects = length(subjects_train)
n_folds = 5

println("Sujetos disponibles en TRAIN: ", n_subjects)

# -----------------------------------------------------------
# Función generadora de folds balanceados
# -----------------------------------------------------------
function generate_subjectwise_folds(subjects::Vector, k::Int=5; seed=104)
    Random.seed!(seed)
    shuffled = Random.shuffle(subjects)

    base_size = div(length(shuffled), k)
    extra = mod(length(shuffled), k)

    folds = Vector{Vector{eltype(subjects)}}()
    start_idx = 1

    for i in 1:k
        fold_size = base_size + (i <= extra ? 1 : 0)
        push!(folds, shuffled[start_idx:start_idx+fold_size-1])
        start_idx += fold_size
    end

    return folds
end

folds = generate_subjectwise_folds(subjects_train, n_folds)

println("\nSujetos por fold:")
for i in 1:length(folds)
    println("Fold $i: ", folds[i])
end

# -----------------------------------------------------------
# Crear los CSV de train/test por fold
# -----------------------------------------------------------

# Crear la carpeta de salida si no existe
mkpath("data_processed/folds")

for i in 1:n_folds
    fold_subjects = folds[i]

    # Elegimos 1 sujeto como test interno (igual que tu script original)
    fold_subjects_shuffled = Random.shuffle(copy(fold_subjects))
    test_subject = fold_subjects_shuffled[1]        # sujeto de test interno
    train_subjects = fold_subjects_shuffled[2:end]  # resto son train

    println("\nFold $i")
    println("  Sujeto de test interno: ", test_subject)
    println("  Sujetos de train: ", train_subjects)

    test_mask  = in.(df_train.subject, Ref([test_subject]))
    train_mask = in.(df_train.subject, Ref(train_subjects))

    df_fold_train = df_train[train_mask, :]
    df_fold_test  = df_train[test_mask, :]

    # Guardar CSVs en data_processed/folds/
    CSV.write("data_processed/folds/fold$(i)_train.csv", df_fold_train)
    CSV.write("data_processed/folds/fold$(i)_test.csv",  df_fold_test)
end

println("\nValidación cruzada individual-wise 5-fold generada correctamente.")
println("Archivos guardados en data_processed/folds/")


Sujetos disponibles en TRAIN: 13

Sujetos por fold:
Fold 1: [20, 10, 12]
Fold 2: [4, 22, 2]
Fold 3: [30, 26, 6]
Fold 4: [14, 18]
Fold 5: [24, 28]

Fold 1
  Sujeto de test interno: 10
  Sujetos de train: [12, 20]

Fold 2
  Sujeto de test interno: 4
  Sujetos de train: [22, 2]

Fold 3
  Sujeto de test interno: 26
  Sujetos de train: [6, 30]

Fold 4
  Sujeto de test interno: 14
  Sujetos de train: [18]

Fold 5
  Sujeto de test interno: 28
  Sujetos de train: [24]

Validación cruzada individual-wise 5-fold generada correctamente.
Archivos guardados en data_processed/folds/


### 7. Normalización Min-Max con un nodo MLJ personalizado

La rúbrica de la práctica exige que la normalización se implemente como un **Nodo de MLJ**, y no como
una operación manual. El objetivo es garantizar que:

- El normalizador se ajusta **únicamente con el conjunto de entrenamiento**.
- El mismo transformador se aplica después sobre validaciones internas y sobre el conjunto de test.
- Se evita cualquier **fuga de información**.
- La normalización pueda integrarse dentro de un **pipeline de MLJ** o del proceso de validación cruzada.

En nuestro entorno concreto, los transformadores predefinidos de MLJ
(`Standardizer`, `FeatureRescaler`, `UnivariateStandardizer`, etc.) no estaban disponibles en la
versión de `MLJModels` instalada.  
Para seguir estrictamente la rúbrica, optamos por implementar un **nodo MLJ propio**, totalmente
compatible con la interfaz de MLJ.

Este nodo (`MyMinMaxScaler`) implementa:

- `fit(model, X)` → calcula los mínimos y máximos por columna únicamente a partir del conjunto *train*  
- `transform(model, X)` → aplica la fórmula del Min-Max scaling  
- Exclusión automática de columnas no numéricas  
- Ignora explícitamente las columnas `subject` y `Activity`, que no deben normalizarse  
- Es robusto ante valores `missing`  

Con este nodo se obtiene un funcionamiento equivalente al Min-Max tradicional,  
pero **respetando la filosofía y requisitos formales de MLJ**.

In [65]:
#-----------------------------------------------------------
# Definición del escalador Min-Max personalizado
#-----------------------------------------------------------
const MMI = MLJModelInterface

struct MyMinMaxScaler <: MMI.Unsupervised
    ignore::Vector{Symbol}
end

MyMinMaxScaler(; ignore = [:subject, :Activity]) = MyMinMaxScaler(ignore)

#-----------------------------------------------------------
# Implementación de fit y transform
#-----------------------------------------------------------
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)

    # 1. columnas realmente numéricas
    numeric_cols = [
        c for c in names(X)
        if !(c in model.ignore) &&
           all(x -> x === missing || x isa Real, X[!, c])
    ]

    mins = Dict{String, Float64}()
    maxs = Dict{String, Float64}()

    for col in numeric_cols
        col_data = collect(skipmissing(X[!, col]))

        if isempty(col_data)
            mins[col] = 0.0
            maxs[col] = 0.0
        else
            mins[col] = minimum(col_data)
            maxs[col] = maximum(col_data)
        end
    end

    fitresult = (
        mins = mins,
        maxs = maxs,
        numeric_cols = numeric_cols
    )

    return fitresult, nothing, nothing
end


function MMI.transform(model::MyMinMaxScaler, fitresult, X)
    X_new = deepcopy(X)

    for col in fitresult.numeric_cols
        minv = fitresult.mins[col]
        maxv = fitresult.maxs[col]

        if maxv != minv
            X_new[!, col] = (X_new[!, col] .- minv) ./ (maxv - minv)
        else
            X_new[!, col] .= 0.0
        end
    end

    return X_new
end

#-----------------------------------------------------------
# Aplicar el escalado Min-Max y guardar los datasets escalados
#-----------------------------------------------------------

# Cargar los datasets de train y test desde data_processed
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)
df_test  = CSV.read("data_processed/dataset_test.csv", DataFrame)

scaler = MyMinMaxScaler(ignore = [:subject, :Activity])

mach = machine(scaler, df_train)
fit!(mach)

df_train_scaled = transform(mach, df_train)
df_test_scaled  = transform(mach, df_test)

CSV.write("data_processed/dataset_train_scaled.csv", df_train_scaled)
CSV.write("data_processed/dataset_test_scaled.csv", df_test_scaled)

[ Info: Training machine(MyMinMaxScaler(ignore = [:subject, :Activity]), …).


"data_processed/dataset_test_scaled.csv"